In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "True"

import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer

from sentence_transformers import SentenceTransformer

from bertopic import BERTopic
from bertopic.dimensionality import BaseDimensionalityReduction
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance

In [ ]:
all_text = pd.read_parquet('clean_text_exploded.parquet').reset_index()
all_docs = all_text['Cleaned_Text'].to_list()
all_docs = [doc if doc.endswith('. ') else doc + '. ' for doc in all_docs]
reduced_embeddings = np.load("reduced_embeddings.npy")

### Replace the 'n_clusters' variable below with the optimal number of clusters determined in the coherence experiments.

In [ ]:
sentence_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device="mps")
ctfidf_model        = ClassTfidfTransformer(reduce_frequent_words=True)
vectorizer_model    = CountVectorizer(stop_words="english", ngram_range=(1,2), min_df=5)
dim_model           = BaseDimensionalityReduction()
representation_model = {
    'keybert': KeyBERTInspired(),
    'mmr':     MaximalMarginalRelevance(diversity=0.3)
}

cluster = KMeans(n_clusters=15, init="k-means++", n_init=5, max_iter=100, algorithm="elkan", random_state=39)

model = BERTopic(low_memory=True, embedding_model=sentence_model, umap_model=dim_model, hdbscan_model=cluster, 
                     vectorizer_model=vectorizer_model, calculate_probabilities=False, ctfidf_model=ctfidf_model, 
                     representation_model=representation_model, verbose=True)

topics, probabilities = model.fit_transform(documents=all_docs, embeddings=reduced_embeddings) 
model.save("model", serialization="safetensors", save_embedding_model=sentence_model, save_ctfidf=True)

In [ ]:
train_idxs = range(len(all_text))
doc_topic = pd.DataFrame({
  'Topic':model.topics_,
  'ID':range(len(model.topics_)),
  'Document': all_text.loc[train_idxs, 'Cleaned_Text']}
) 
doc_topic.to_csv('topic_model_raw.csv')

In [ ]:
repr_docs, _, _, _=  model._extract_representative_docs(
    model.c_tf_idf_, 
    doc_topic,
    model.topic_representations_,
    nr_repr_docs=250
)
model.representative_docs_ = repr_docs
model.get_topic_info().to_csv('topic_model_representative.csv')